# Chest Disease Detection: CheXFound + GLoRI Adaptation

This notebook adapts the methodology in `src/CheXFound` to the local chest disease multilabel task.

CheXFound's key idea is not just "use a ViT". It freezes a chest X-ray foundation encoder, extracts patch tokens from the last transformer blocks, then trains a lightweight GLoRI head where disease-specific query tokens attend to local patch features.

For this competition task, the notebook uses:

- The same data/template handling as the MedCLIP baseline.
- CheXFound-style image preprocessing with per-image rescaling and ImageNet normalization.
- A frozen patch-token encoder.
- A trainable GLoRI multilabel head with one query per disease label.
- `BCEWithLogitsLoss` with clipped positive weights.
- Threshold tuning for sample-average F1.

Set `CHEXFOUND_CONFIG` and `CHEXFOUND_WEIGHTS` to use a native CheXFound checkpoint. Without those files, the notebook uses a timm ViT fallback while preserving the GLoRI patch-query training method.

In [ ]:
# Dependencies. The CheXFound source tree is used directly from src/CheXFound.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("XFORMERS_DISABLED", "1")

required = {
    "timm": "timm",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    except subprocess.CalledProcessError as exc:
        print("Dependency install failed. Continuing; preinstalled packages may still be enough.")
        print(exc)

import tomllib


def find_repo_root():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "pyproject.toml").exists():
            return parent.resolve()
    return Path.cwd().resolve()


def load_chest_config():
    pyproject = find_repo_root() / "pyproject.toml"
    if not pyproject.exists():
        return {}
    with pyproject.open("rb") as f:
        data = tomllib.load(f)
    return data.get("tool", {}).get("spai", {}).get("chest", {})


REPO_ROOT = find_repo_root()
CHEST_CONFIG = load_chest_config()
print("Repo root:", REPO_ROOT)
print("Chest config:", CHEST_CONFIG)


def resolve_config_path(value):
    if not value:
        return None
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()


def find_chexfound_root():
    manual = os.environ.get("CHEXFOUND_ROOT") or CHEST_CONFIG.get("chexfound_root")
    if manual:
        root = resolve_config_path(manual)
        if root.exists():
            return root
        raise FileNotFoundError(f"Configured CheXFound root does not exist: {root}")

    starts = [Path.cwd(), REPO_ROOT, Path.cwd().parent, Path.cwd().parent.parent]
    for start in starts:
        for parent in [start, *start.parents]:
            candidate = parent / "src" / "CheXFound"
            if (candidate / "chexfound" / "eval" / "classification" / "glori.py").exists():
                return candidate.resolve()

    raise FileNotFoundError(
        "CheXFound source not found. Configure tool.spai.chest.chexfound_root in pyproject.toml "
        "or set CHEXFOUND_ROOT."
    )


CHEXFOUND_ROOT = find_chexfound_root()
sys.path.insert(0, str(CHEXFOUND_ROOT))
print("Using CheXFound source:", CHEXFOUND_ROOT)

In [ ]:
import glob
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Re-apply CheXFound path discovery here too, so this cell works even if the setup cell
# was skipped or the kernel state was reset.
try:
    GLoRI
except NameError:
    try:
        from chexfound.eval.classification.glori import GLoRI
    except ModuleNotFoundError:
        import sys
        import tomllib
        from torch import Tensor
        from typing import Optional
        from torch.nn.modules.transformer import _get_activation_fn

        def _repo_root_for_import():
            for parent in [Path.cwd(), *Path.cwd().parents]:
                if (parent / "pyproject.toml").exists():
                    return parent.resolve()
            return Path.cwd().resolve()

        def _configured_chexfound_root():
            repo = _repo_root_for_import()
            pyproject = repo / "pyproject.toml"
            configured = None
            if pyproject.exists():
                with pyproject.open("rb") as f:
                    configured = tomllib.load(f).get("tool", {}).get("spai", {}).get("chest", {}).get("chexfound_root")
            candidates = []
            if os.environ.get("CHEXFOUND_ROOT"):
                candidates.append(Path(os.environ["CHEXFOUND_ROOT"]).expanduser())
            if configured:
                candidates.append(Path(configured).expanduser() if Path(configured).is_absolute() else repo / configured)
            for parent in [Path.cwd(), repo, *Path.cwd().parents]:
                candidates.append(parent / "src" / "CheXFound")
            for candidate in candidates:
                if (candidate / "chexfound" / "eval" / "classification" / "glori.py").exists():
                    return candidate.resolve()
            return None

        root = _configured_chexfound_root()
        if root is not None:
            sys.path.insert(0, str(root))
            from chexfound.eval.classification.glori import GLoRI
            print("Imported GLoRI from:", root)
        else:
            print("CheXFound package not found; using inline GLoRI fallback.")

            def create_mldecoder_input(x_tokens_list, use_n_blocks):
                intermediate_output = x_tokens_list[-use_n_blocks:]
                output = torch.cat([patch_token for patch_token, _ in intermediate_output], dim=-1)
                cls = torch.cat([cls_tok for _, cls_tok in intermediate_output], dim=-1)
                return output.float(), cls.float()

            class TransformerDecoderLayerOptimal(nn.Module):
                def __init__(self, d_model, nhead=8, dim_feedforward=2048, dropout=0.1, activation="relu", layer_norm_eps=1e-5):
                    super().__init__()
                    self.norm1 = nn.LayerNorm(d_model, eps=layer_norm_eps)
                    self.dropout = nn.Dropout(dropout)
                    self.dropout1 = nn.Dropout(dropout)
                    self.dropout2 = nn.Dropout(dropout)
                    self.dropout3 = nn.Dropout(dropout)
                    self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
                    self.linear1 = nn.Linear(d_model, dim_feedforward)
                    self.linear2 = nn.Linear(dim_feedforward, d_model)
                    self.norm2 = nn.LayerNorm(d_model, eps=layer_norm_eps)
                    self.norm3 = nn.LayerNorm(d_model, eps=layer_norm_eps)
                    self.activation = _get_activation_fn(activation)

                def forward(self, tgt: Tensor, memory: Tensor, tgt_mask: Optional[Tensor] = None,
                            memory_mask: Optional[Tensor] = None, tgt_key_padding_mask: Optional[Tensor] = None,
                            memory_key_padding_mask: Optional[Tensor] = None, return_attention=False) -> Tensor:
                    tgt = self.norm1(tgt + self.dropout1(tgt))
                    tgt2, attn = self.multihead_attn(tgt, memory, memory, average_attn_weights=False)
                    tgt = self.norm2(tgt + self.dropout2(tgt2))
                    tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
                    tgt = self.norm3(tgt + self.dropout3(tgt2))
                    return (tgt, attn) if return_attention else tgt

            class TransformerDecoder(nn.TransformerDecoder):
                def forward(self, tgt: Tensor, memory: Tensor, tgt_mask: Optional[Tensor] = None,
                            memory_mask: Optional[Tensor] = None, tgt_key_padding_mask: Optional[Tensor] = None,
                            memory_key_padding_mask: Optional[Tensor] = None, return_attention=False) -> Tensor:
                    output = tgt
                    for i, mod in enumerate(self.layers):
                        if i < len(self.layers) - 1:
                            output = mod(output, memory, tgt_mask=tgt_mask, memory_mask=memory_mask,
                                         tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=memory_key_padding_mask)
                        else:
                            result = mod(output, memory, tgt_mask=tgt_mask, memory_mask=memory_mask,
                                         tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=memory_key_padding_mask,
                                         return_attention=True)
                            output, attn = result
                            if return_attention:
                                return attn
                    return self.norm(output) if self.norm is not None else output

            class GLoRI(nn.Module):
                def __init__(self, num_classes, decoder_embedding=768, initial_num_features=2048,
                             use_n_blocks=4, multiview=False, cat_cls=False):
                    super().__init__()
                    num_queries = num_classes
                    decoder_embedding = 768 if decoder_embedding < 0 else decoder_embedding
                    self.embed_standart = nn.Linear(initial_num_features, decoder_embedding)
                    self.query_embed = nn.Embedding(num_queries, decoder_embedding)
                    self.query_embed.requires_grad_(False)
                    layer = TransformerDecoderLayerOptimal(d_model=decoder_embedding, dim_feedforward=2048, dropout=0.1)
                    self.decoder = TransformerDecoder(layer, num_layers=1)
                    self.duplicate_pooling = nn.Parameter(torch.Tensor(num_queries, decoder_embedding + (initial_num_features if cat_cls else 0), 1))
                    self.duplicate_pooling_bias = nn.Parameter(torch.Tensor(num_classes))
                    torch.nn.init.xavier_normal_(self.duplicate_pooling)
                    torch.nn.init.constant_(self.duplicate_pooling_bias, 0)
                    self.num_classes = num_classes
                    self.use_n_blocks = use_n_blocks
                    self.cat_cls = cat_cls

                def forward(self, x, return_attention=False):
                    x, cls = create_mldecoder_input(x[0], use_n_blocks=self.use_n_blocks)
                    memory = torch.relu(self.embed_standart(x)).transpose(0, 1)
                    bs = memory.shape[1]
                    tgt = self.query_embed.weight.unsqueeze(1).expand(-1, bs, -1)
                    h = self.decoder(tgt, memory, return_attention=return_attention)
                    if return_attention:
                        return h
                    h = h.transpose(0, 1)
                    if self.cat_cls:
                        h = torch.cat([cls.unsqueeze(1).repeat(1, h.shape[1], 1), h], dim=-1)
                    logits = torch.einsum("bqd,qdk->bqk", h, self.duplicate_pooling).squeeze(-1)
                    return logits[:, :self.num_classes] + self.duplicate_pooling_bias

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)

SEED = int(os.environ.get("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)


def mean_column_auc(y_true, y_prob):
    aucs = []
    for c in range(y_true.shape[1]):
        if len(np.unique(y_true[:, c])) > 1:
            aucs.append(roc_auc_score(y_true[:, c], y_prob[:, c]))
    return float(np.mean(aucs)) if aucs else float("nan")


def samples_f1(y_true, y_bin):
    return f1_score(y_true, y_bin, average="samples", zero_division=0)


def to_binary(prob, thr):
    b = (prob >= thr).astype(np.int64)
    empty = b.sum(axis=1) == 0
    if empty.any():
        b[empty, prob[empty].argmax(axis=1)] = 1
    return b


def best_threshold(y_true, prob, lo=0.05, hi=0.60, step=0.01):
    ths = np.arange(lo, hi + 1e-9, step)
    f1s = np.array([samples_f1(y_true, to_binary(prob, t)) for t in ths])
    i = int(f1s.argmax())
    return float(ths[i]), float(f1s[i]), ths, f1s

In [ ]:
COMP_NAME = os.environ.get("COMP_NAME", "individual-test-chest-disease-detection")
CONFIG_DATA_ROOT = resolve_config_path(CHEST_CONFIG.get("data_root"))


def candidate_roots():
    roots = []
    if CONFIG_DATA_ROOT is not None:
        roots.append(CONFIG_DATA_ROOT)
    roots += [Path("/kaggle/input") / COMP_NAME]
    roots += [Path(p) for p in glob.glob("/kaggle/input/*")]
    roots += [REPO_ROOT / "dataset" / COMP_NAME, Path("dataset") / COMP_NAME, Path("dataset"), Path("comp_data"), Path(".")]
    roots += [Path(p) for p in glob.glob("dataset/*")]
    seen = set()
    for r in roots:
        key = str(r.resolve()) if r.exists() else str(r)
        if key not in seen:
            seen.add(key)
            yield r


def find_data_root():
    manual = os.environ.get("DATA_ROOT")
    if manual:
        r = Path(manual)
        if (r / "train.csv").exists():
            return r
        raise FileNotFoundError(f"DATA_ROOT does not contain train.csv: {r}")

    for r in candidate_roots():
        if (r / "train.csv").exists():
            return r

    zips = sorted(Path(".").glob("*.zip"), key=lambda p: p.stat().st_size, reverse=True)
    if zips:
        import zipfile
        out = Path("dataset") / COMP_NAME
        out.mkdir(parents=True, exist_ok=True)
        print("Extracting", zips[0], "to", out)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(out)
        if (out / "train.csv").exists():
            return out

    raise FileNotFoundError("Could not find train.csv. Set DATA_ROOT or place competition files under dataset/.")


def find_images_dir(root):
    root = Path(root)
    for rel in ["images/images", "images", "."]:
        d = root / rel
        if list(d.glob("*.jpg")) or list(d.glob("*.png")):
            return d
    for d in root.rglob("*"):
        if d.is_dir() and (list(d.glob("*.jpg")) or list(d.glob("*.png"))):
            return d
    raise FileNotFoundError(f"No image directory found under {root}")


DATA_ROOT = find_data_root()
IMAGES_DIR = find_images_dir(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("IMAGES_DIR:", IMAGES_DIR)

In [ ]:
train = pd.read_csv(DATA_ROOT / "train.csv")
LABELS = [c for c in train.columns if c != "filename"]
N_CLASSES = len(LABELS)
assert N_CLASSES > 0, "Expected train.csv with filename plus disease label columns."
train[LABELS] = train[LABELS].fillna(0).clip(0, 1).astype(np.float32)

image_paths = {}
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for p in IMAGES_DIR.glob(ext):
        image_paths[p.name] = p
train = train[train["filename"].isin(image_paths)].reset_index(drop=True)
assert len(train) > 0, "No train filenames match images on disk."

submission_candidates = [DATA_ROOT / "test_submission.csv", DATA_ROOT / "sample_submission.csv"]
submission_path = next((p for p in submission_candidates if p.exists()), None)
assert submission_path is not None, "Could not find test_submission.csv or sample_submission.csv."
SAMPLE_SUB = pd.read_csv(submission_path)
missing_cols = [c for c in LABELS if c not in SAMPLE_SUB.columns]
assert "filename" in SAMPLE_SUB.columns and not missing_cols, f"Bad submission template. Missing: {missing_cols}"

PREDICT_MASK = SAMPLE_SUB[LABELS].isna().all(axis=1)
test_files = SAMPLE_SUB.loc[PREDICT_MASK, "filename"].tolist()

print(f"Labels ({N_CLASSES}):", LABELS)
print("Train rows:", len(train))
print("Submission template:", SAMPLE_SUB.shape)
print("Rows to predict:", len(test_files), "| pre-filled rows:", int((~PREDICT_MASK).sum()))
train.head()

In [ ]:
pos_rate = train[LABELS].mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
pos_rate.plot(kind="bar", ax=ax)
ax.set_ylabel("positive rate")
ax.set_title("Label positive rate")
plt.tight_layout()
plt.show()

cooc = train[LABELS].T.dot(train[LABELS]).astype(int)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cooc, cmap="mako", ax=ax)
ax.set_title("Label co-occurrence")
plt.tight_layout()
plt.show()

In [ ]:
IMG_SIZE = int(os.environ.get("IMG_SIZE", "512"))
RESIZE_SIZE = int(os.environ.get("RESIZE_SIZE", str(IMG_SIZE)))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class MaybeToTensor(transforms.ToTensor):
    def __call__(self, pic):
        if isinstance(pic, torch.Tensor):
            return pic
        return super().__call__(pic)


class RescaleImage:
    def __call__(self, image):
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image)
        if not torch.is_tensor(image):
            raise TypeError("Input should be numpy array or torch tensor")
        image = image.float()
        flat = image.reshape(image.shape[0], -1)
        min_val = flat.min(dim=1)[0].reshape(-1, 1, 1)
        max_val = flat.max(dim=1)[0].reshape(-1, 1, 1)
        denom = (max_val - min_val).clamp_min(1e-6)
        return (image - min_val) / denom


def build_transforms():
    # CheXFound-style evaluation transform: resize, center crop, tensor, per-image rescale, ImageNet normalize.
    eval_tf = transforms.Compose([
        transforms.Resize(RESIZE_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMG_SIZE),
        MaybeToTensor(),
        RescaleImage(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    # Keep augmentation anatomy-safe: no horizontal flip for CXR laterality.
    train_tf = transforms.Compose([
        transforms.Resize(RESIZE_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.97, 1.03)),
        transforms.ColorJitter(brightness=0.08, contrast=0.08),
        MaybeToTensor(),
        RescaleImage(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_tf, eval_tf


class CXRDataset(Dataset):
    def __init__(self, files, labels, transform, image_map):
        self.files = list(files)
        self.labels = labels
        self.transform = transform
        self.image_map = image_map

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        image = Image.open(self.image_map[filename]).convert("RGB")
        image = self.transform(image)
        if self.labels is None:
            return image, filename
        return image, torch.tensor(self.labels[idx], dtype=torch.float32)


train_tf, eval_tf = build_transforms()
print("Image size:", IMG_SIZE)

In [ ]:
def load_native_chexfound_encoder():
    config_path = os.environ.get("CHEXFOUND_CONFIG") or CHEST_CONFIG.get("chexfound_config")
    weights_path = os.environ.get("CHEXFOUND_WEIGHTS") or CHEST_CONFIG.get("chexfound_weights")
    config_path = resolve_config_path(config_path)
    weights_path = resolve_config_path(weights_path)
    if not config_path or not weights_path:
        return None
    if not config_path.exists() or not weights_path.exists():
        print("Configured native CheXFound checkpoint files are missing:")
        print("  config:", config_path)
        print("  weights:", weights_path)
        return None

    try:
        from omegaconf import OmegaConf
    except ImportError:
        print("omegaconf is not installed, so native CheXFound config loading is unavailable.")
        print("Install omegaconf or leave CHEXFOUND_CONFIG/CHEXFOUND_WEIGHTS unset to use the timm token fallback.")
        return None
    from chexfound.models import build_model_from_cfg

    cfg = OmegaConf.load(config_path)
    model, _ = build_model_from_cfg(cfg, only_teacher=True)
    state = torch.load(weights_path, map_location="cpu")
    state_dict = state.get("teacher", state)

    cleaned = {}
    for k, v in state_dict.items():
        if k.startswith("backbone"):
            parts = k.split(".")
            if "blocks" in k:
                new_key = ".".join([parts[1], *parts[3:]])
            else:
                new_key = ".".join(parts[1:])
        else:
            new_key = k
        cleaned[new_key] = v

    msg = model.load_state_dict(cleaned, strict=False)
    model.eval()
    print("Loaded native CheXFound encoder:", msg)
    return model


class TimmTokenEncoder(nn.Module):
    """timm fallback exposing a CheXFound-like get_intermediate_layers API."""
    def __init__(self, model_name):
        super().__init__()
        import timm
        self.model = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.model_name = model_name
        self.embed_dim = getattr(self.model, "num_features", None) or getattr(self.model, "embed_dim", None)
        if self.embed_dim is None:
            raise ValueError(f"Cannot infer embedding dim for {model_name}")

    def get_intermediate_layers(self, x, n=4, return_class_token=True):
        out = self.model.forward_features(x)
        if isinstance(out, dict):
            if "x_norm_patchtokens" in out and "x_norm_clstoken" in out:
                patch = out["x_norm_patchtokens"]
                cls = out["x_norm_clstoken"]
            else:
                out = next(v for v in out.values() if torch.is_tensor(v))
        if torch.is_tensor(out):
            if out.ndim == 4:
                patch = out.flatten(2).transpose(1, 2)
                cls = patch.mean(dim=1)
            elif out.ndim == 3:
                cls = out[:, 0]
                patch = out[:, 1:]
            elif out.ndim == 2:
                cls = out
                patch = out.unsqueeze(1)
            else:
                raise ValueError(f"Unsupported timm feature shape: {tuple(out.shape)}")
        return tuple((patch, cls) for _ in range(n)) if return_class_token else tuple(patch for _ in range(n))

    def forward(self, x):
        feats = self.get_intermediate_layers(x, n=1, return_class_token=True)
        return feats[-1][1]


def build_patch_encoder():
    native = load_native_chexfound_encoder()
    if native is not None:
        return native, "native-chexfound"

    requested = os.environ.get("TIMM_TOKEN_BACKBONE")
    candidates = [requested] if requested else []
    candidates += [
        "vit_base_patch16_224.dino",
        "vit_base_patch16_224.augreg2_in21k_ft_in1k",
        "vit_base_patch16_224.augreg_in21k_ft_in1k",
        "deit_base_patch16_224.fb_in1k",
    ]
    seen = set()
    errors = []
    for model_name in [c for c in candidates if c and not (c in seen or seen.add(c))]:
        try:
            enc = TimmTokenEncoder(model_name)
            print("Using timm token fallback:", model_name)
            return enc, f"timm:{model_name}"
        except Exception as exc:
            errors.append((model_name, repr(exc)))
            print(f"timm fallback failed for {model_name}: {type(exc).__name__}: {exc}")
    raise RuntimeError("No patch-token encoder could be loaded: " + str(errors))


N_LAST_BLOCKS = int(os.environ.get("N_LAST_BLOCKS", "4"))
FREEZE_ENCODER = os.environ.get("FREEZE_ENCODER", "1") == "1"

encoder, ENCODER_NAME = build_patch_encoder()
encoder = encoder.to(DEVICE)
if FREEZE_ENCODER:
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad_(False)

with torch.no_grad():
    sample = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    sample_output = encoder.get_intermediate_layers(sample, n=N_LAST_BLOCKS, return_class_token=True)
    patch_dim = sample_output[-1][0].shape[-1] * N_LAST_BLOCKS

print("Encoder:", ENCODER_NAME)
print("Patch-token dim into GLoRI:", patch_dim)
print("Patch token count:", sample_output[-1][0].shape[1])

In [ ]:
class CheXFoundGLoRIModel(nn.Module):
    def __init__(self, encoder, n_classes, n_last_blocks=4, decoder_dim=768, cat_cls=False, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.n_last_blocks = n_last_blocks
        self.freeze_encoder = freeze_encoder
        self.glori = GLoRI(
            num_classes=n_classes,
            decoder_embedding=decoder_dim,
            initial_num_features=patch_dim,
            use_n_blocks=n_last_blocks,
            multiview=False,
            cat_cls=cat_cls,
        )

    def forward(self, x):
        if self.freeze_encoder:
            self.encoder.eval()
            with torch.no_grad():
                features = self.encoder.get_intermediate_layers(
                    x, n=self.n_last_blocks, return_class_token=True
                )
        else:
            features = self.encoder.get_intermediate_layers(
                x, n=self.n_last_blocks, return_class_token=True
            )
        return self.glori([features])


model = CheXFoundGLoRIModel(
    encoder=encoder,
    n_classes=N_CLASSES,
    n_last_blocks=N_LAST_BLOCKS,
    decoder_dim=int(os.environ.get("GLORI_DECODER_DIM", "768")),
    cat_cls=os.environ.get("GLORI_CAT_CLS", "0") == "1",
    freeze_encoder=FREEZE_ENCODER,
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

In [ ]:
EPOCHS = int(os.environ.get("EPOCHS", "8"))
BATCH = int(os.environ.get("BATCH", "8" if IMG_SIZE >= 512 else "32"))
WORKERS = int(os.environ.get("WORKERS", "0" if DEVICE.type == "mps" else "2"))
VAL_SIZE = float(os.environ.get("VAL_SIZE", "0.10"))

all_y = train[LABELS].values.astype("float32")
label_count_bin = np.clip(all_y.sum(axis=1).astype(int), 0, 3)
unique, counts = np.unique(label_count_bin, return_counts=True)
stratify = label_count_bin if len(unique) > 1 and counts.min() >= 2 else None

tr_idx, va_idx = train_test_split(
    np.arange(len(train)), test_size=VAL_SIZE, random_state=SEED, stratify=stratify
)

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    CXRDataset(train["filename"].values[tr_idx], all_y[tr_idx], train_tf, image_paths),
    batch_size=BATCH, shuffle=True, num_workers=WORKERS, pin_memory=pin_memory,
)
val_loader = DataLoader(
    CXRDataset(train["filename"].values[va_idx], all_y[va_idx], eval_tf, image_paths),
    batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=pin_memory,
)

pos = all_y[tr_idx].sum(axis=0)
neg = len(tr_idx) - pos
pos_weight = np.clip(neg / np.clip(pos, 1, None), 1.0, 10.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

param_groups = [{"params": model.glori.parameters(), "lr": float(os.environ.get("GLORI_LR", "1e-4"))}]
if not FREEZE_ENCODER:
    param_groups.append({"params": model.encoder.parameters(), "lr": float(os.environ.get("ENCODER_LR", "1e-6"))})
optimizer = torch.optim.AdamW(param_groups, weight_decay=float(os.environ.get("WEIGHT_DECAY", "1e-4")))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Train size:", len(tr_idx), "| Val size:", len(va_idx))
print("Batch:", BATCH, "| Workers:", WORKERS, "| Epochs:", EPOCHS)

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    probs, targets = [], []
    for x, y in tqdm(loader, desc="validate", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        probs.append(torch.sigmoid(logits).float().cpu().numpy())
        targets.append(y.numpy())
    probs = np.concatenate(probs)
    targets = np.concatenate(targets)
    auc = mean_column_auc(targets, probs)
    thr, f1, _, _ = best_threshold(targets, probs)
    return auc, f1, thr, probs, targets


history = []
best_f1 = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    if FREEZE_ENCODER:
        model.encoder.eval()
    running_loss = 0.0
    seen = 0
    started = time.time()
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")

    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x.size(0)
        seen += x.size(0)
        pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

    scheduler.step()
    val_auc, val_f1, val_thr, _, _ = evaluate(val_loader)
    history.append({"epoch": epoch, "loss": running_loss / len(tr_idx), "val_auc": val_auc, "val_f1": val_f1, "thr": val_thr})
    print(
        f"epoch {epoch}/{EPOCHS} | loss {running_loss / len(tr_idx):.4f} | "
        f"val_auc {val_auc:.4f} | val_sample_f1 {val_f1:.4f} | thr {val_thr:.2f} | {time.time() - started:.0f}s"
    )
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

hist_df = pd.DataFrame(history)
print("Best validation sample-F1:", best_f1)
hist_df

In [ ]:
val_auc, _, _, val_prob, val_true = evaluate(val_loader)
BEST_THR, BEST_F1, ths, f1s = best_threshold(val_true, val_prob)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ths, f1s)
ax.axvline(BEST_THR, color="red", linestyle="--", label=f"best = {BEST_THR:.2f}")
ax.set_xlabel("threshold")
ax.set_ylabel("sample-average F1")
ax.set_title("Validation threshold tuning")
ax.legend()
plt.tight_layout()
plt.show()

val_bin = to_binary(val_prob, BEST_THR)
per_class_f1 = pd.Series(
    {label: f1_score(val_true[:, i], val_bin[:, i], zero_division=0) for i, label in enumerate(LABELS)}
).sort_values()
print(f"Validation sample-average F1 = {BEST_F1:.4f} @ threshold {BEST_THR:.2f}")
print(f"Validation mean column AUC = {val_auc:.4f}")
per_class_f1.plot(kind="barh", figsize=(8, max(4, 0.35 * len(per_class_f1))), title="Per-class F1")
plt.tight_layout()
plt.show()
per_class_f1

In [ ]:
test_image_paths = dict(image_paths)
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for p in IMAGES_DIR.glob(ext):
        test_image_paths[p.name] = p
missing_test = [f for f in test_files if f not in test_image_paths]
assert not missing_test, f"Missing test images, first examples: {missing_test[:5]}"

test_loader = DataLoader(
    CXRDataset(test_files, None, eval_tf, test_image_paths),
    batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=pin_memory,
)


@torch.no_grad()
def predict(loader):
    model.eval()
    probs, names = [], []
    for x, filenames in tqdm(loader, desc="predict"):
        x = x.to(DEVICE, non_blocking=True)
        probs.append(torch.sigmoid(model(x)).float().cpu().numpy())
        names.extend(list(filenames))
    if not probs:
        return np.zeros((0, N_CLASSES), dtype=np.float32), []
    return np.concatenate(probs), names


test_prob, pred_names = predict(test_loader)
pred_bin = to_binary(test_prob, BEST_THR) if len(pred_names) else np.zeros((0, N_CLASSES), dtype=np.int64)
pred_map = {name: row for name, row in zip(pred_names, pred_bin)}

submission = SAMPLE_SUB.copy()
for idx in submission.index[PREDICT_MASK]:
    submission.loc[idx, LABELS] = pred_map[submission.at[idx, "filename"]]
submission[LABELS] = submission[LABELS].astype(int)

assert list(submission.columns) == list(SAMPLE_SUB.columns)
assert submission[LABELS].isna().sum().sum() == 0
assert submission[LABELS].isin([0, 1]).all().all()

out_path = Path("submission_chexfound_glori.csv")
submission.to_csv(out_path, index=False)
print("Saved", out_path.resolve())
print(f"encoder={ENCODER_NAME} | val_sample_f1={BEST_F1:.4f} | threshold={BEST_THR:.2f}")
submission.head()

In [ ]:
# Optional: visualize GLoRI disease-query attention for one validation image.
# This works because GLoRI returns attention from disease queries to patch tokens.
VISUALIZE_ATTENTION = os.environ.get("VISUALIZE_ATTENTION", "0") == "1"
if VISUALIZE_ATTENTION:
    model.eval()
    filename = train["filename"].values[va_idx[0]]
    image = Image.open(image_paths[filename]).convert("RGB")
    x = eval_tf(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        features = model.encoder.get_intermediate_layers(x, n=N_LAST_BLOCKS, return_class_token=True)
        attn = model.glori([features], return_attention=True)
        logits = model.glori([features])
        prob = torch.sigmoid(logits)[0].detach().cpu().numpy()

    # attn shape: batch, heads, disease_queries, patch_tokens
    attn = attn[0].detach().cpu().numpy()
    label_idx = int(prob.argmax())
    token_count = attn.shape[-1]
    side = int(np.sqrt(token_count))
    heat = attn[:, label_idx, :].mean(axis=0)
    if side * side == token_count:
        heat = heat.reshape(side, side)
        heat = torch.tensor(heat)[None, None]
        heat = torch.nn.functional.interpolate(heat, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0].numpy()
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(image.resize((IMG_SIZE, IMG_SIZE)), cmap="gray")
        ax[0].set_title(filename)
        ax[0].axis("off")
        ax[1].imshow(image.resize((IMG_SIZE, IMG_SIZE)), cmap="gray")
        ax[1].imshow(heat, cmap="magma", alpha=0.45)
        ax[1].set_title(f"{LABELS[label_idx]} p={prob[label_idx]:.2f}")
        ax[1].axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print("Patch token grid is not square; cannot reshape attention map.")